# Track 1 · Step 2: clean SeaDronesSee and UAVDT

AI251P Capstone, Phase 1 (M1, 12 marks: *"duplicates and corrupt files are handled"*). Owner: Abdulmohsen Alharthi.

Input: the archives that Step 1 put in `01_Data/Raw/`. Nothing in `Raw/` is changed.

What this notebook does, for each dataset:
1. **Corrupt files**: decodes every image; unreadable files are removed with their boxes.
2. **Exact duplicates**: md5 of every file; extra copies are removed.
3. **Near duplicates across flights/sequences**: perceptual hashes (pHash + dHash). Near-identical frames inside one video are expected and are handled by the group split in Step 3. Near-identical frames in *two different* videos/sequences would leak, so those videos are linked into one **split group**.
4. **Boxes**: zero-size boxes removed, boxes clipped to the image, exact duplicate boxes removed, boxes rescaled if the image file size differs from the annotation (possible in the compressed release), and a `crop_ok` flag for classification (minimum side, occlusion).
5. SeaDronesSee only: **altitude** read from every metadata key variant (`altitude`, `height_above_takeoff(meter)`), and the **`ignored`** category (id 0) kept as `iscrowd=1` ignore regions for detection, excluded from classification.
6. UAVDT only: frames at **5 fps** (1 in 6), ignore regions, sequence attributes.

Outputs (team naming rule `YYYY-MM-DD_Topic_vN`):
- `01_Data/Processed/SeaDronesSee/…_SDSClean_vN.json` (COCO, train+val pooled; the official split is kept only as a field, because it leaks)
- `01_Data/Processed/UAVDT/…_UAVDTBoxes_vN.csv.gz`, `…_UAVDTFrames_vN.csv`
- `01_Data/Processed/…_CleaningLog_vN.csv` (every removed or changed item and why)
- `01_Data/Processed/…_SDSGroups_vN.csv`, `…_UAVDTGroups_vN.csv` (class counts per split group: the input to Step 3)
- `01_Data/…_Step2Summary_vN.json`

Runtime: CPU. Hashing about 14k SeaDronesSee images and 41k UAVDT frames takes a while; progress is printed.

## 0. Settings

In [1]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
DATA_DIR  = f'{DRIVE_PROJECT}/01_Data'
RAW_DIR   = f'{DATA_DIR}/Raw'
PROC_DIR  = f'{DATA_DIR}/Processed'
LOCAL     = '/content/data'
SDS_LOCAL, UAVDT_LOCAL = f'{LOCAL}/seadronessee', f'{LOCAL}/uavdt'
SEED = 42

# Cleaning thresholds (all reported in the dataset card)
MIN_CROP_SIDE    = 8      # px: boxes smaller than this on either side are kept for detection but not cropped
DUP_BOX_IOU      = 0.95   # same image, same class, IoU above this -> duplicate box
NEAR_DUP_BITS    = 4      # max Hamming distance on BOTH 64-bit pHash and dHash to call two images near-identical
MIN_PAIRS_TO_LINK = 3     # two videos/sequences are linked only if at least this many frame pairs are near-identical
UAVDT_FRAME_STEP = 6      # 30 fps -> 5 fps
# UAVDT toolkit levels (check README in UAV-benchmark-MOTD_v1.0): occlusion 1=no ... 4=large; out-of-view 1=no ... 3=medium
CROP_MAX_OCCLUSION, CROP_MAX_OUT_OF_VIEW = 3, 2
N_WORKERS = 8

## 1. Mount Drive, output names

In [2]:
import os, re, json, time, random, hashlib, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir(DATA_DIR), f'Not found: {DATA_DIR}. Mount Drive with the KAUST account.'
for d in (PROC_DIR, f'{PROC_DIR}/SeaDronesSee', f'{PROC_DIR}/UAVDT', LOCAL): os.makedirs(d, exist_ok=True)

TODAY = time.strftime('%Y-%m-%d')
_done = [int(m.group(1)) for f in os.listdir(DATA_DIR) if (m := re.match(rf'{TODAY}_Step2Summary_v(\d+)\.json$', f))]
VERSION = max(_done, default=0) + 1
def out_name(topic, ext): return f'{TODAY}_{topic}_v{VERSION}.{ext}'
print('outputs of this run use version', VERSION)
random.seed(SEED)

Mounted at /content/drive
outputs of this run use version 1


## 2. Extract the raw archives (skipped if already done in this runtime)

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0: print(r.stderr[-2000:]); raise RuntimeError(cmd)
    return r

def extract(archive, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    marker = dest / f'.extracted_{Path(archive).name}.done'
    if marker.exists(): return
    sh(f'unzip -q -o "{archive}" -d "{dest}"'); marker.touch()

def extract_nested(dest):
    for z in sorted(Path(dest).rglob('*.zip')): extract(z, z.parent)

for name, local in [('SeaDronesSee', SDS_LOCAL), ('UAVDT', UAVDT_LOCAL)]:
    zips = sorted(Path(RAW_DIR, name).glob('*.zip'))
    assert zips, f'No archives in Raw/{name}: run Step 1 first.'
    for z in zips: print('extracting', z.name); extract(z, local)
    extract_nested(local)
print('done')

extracting SeaDronesSee_ODv2_CompressedVersion.zip
extracting M_attr.zip
extracting UAV-benchmark-M.zip
extracting UAV-benchmark-MOTD_v1.0.zip
done


## 3. Image scan helper: decode, md5, perceptual hashes

In [ ]:
!pip -q install imagehash
import numpy as np, pandas as pd, imagehash
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.width', 160)

def scan_one(path, want_hash=True):
    """Fully decode the image (at reduced scale for JPEG, which still reads the whole stream),
    return size, md5 and 64-bit pHash / dHash as ints."""
    out = {'path': str(path), 'readable': False, 'error': None, 'file_w': None, 'file_h': None,
           'md5': None, 'phash': None, 'dhash': None}
    try:
        with open(path, 'rb') as f: data = f.read()
        out['md5'] = hashlib.md5(data).hexdigest()
        with Image.open(path) as im:
            out['file_w'], out['file_h'] = im.size
            im.draft('RGB', (512, 512))      # JPEG: decode at 1/2..1/8 scale, fast
            im = im.convert('RGB'); im.load()
            if want_hash:
                out['phash'] = int(str(imagehash.phash(im)), 16)
                out['dhash'] = int(str(imagehash.dhash(im)), 16)
        out['readable'] = True
    except Exception as e:
        out['error'] = repr(e)[:120]
    return out

def scan(paths, want_hash=True, label=''):
    paths = list(paths); res = []; t0 = time.time()
    with ThreadPoolExecutor(N_WORKERS) as ex:
        for i, r in enumerate(ex.map(lambda p: scan_one(p, want_hash), paths), 1):
            res.append(r)
            if i % 2000 == 0 or i == len(paths):
                print(f'  {label} {i}/{len(paths)}  {time.time() - t0:.0f}s')
    return pd.DataFrame(res)

# Hamming distance on 64-bit hashes, vectorised
_POP8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def hamming_rows(a, B):
    x = np.bitwise_xor(np.uint64(a), B)
    return _POP8[x.view(np.uint8).reshape(-1, 8)].sum(1)

def near_dup_pairs(df, group_col, max_bits=NEAR_DUP_BITS):
    """Near-identical pairs (pHash AND dHash within max_bits).
    Returns (number of pairs inside one group, DataFrame of pairs across groups).
    Pairs inside a group are only counted: consecutive frames of one video are near-identical by nature."""
    ok = df.dropna(subset=['phash', 'dhash']).reset_index(drop=True)
    P = np.array([int(v) for v in ok.phash], dtype=np.uint64)
    D = np.array([int(v) for v in ok.dhash], dtype=np.uint64)
    g = ok[group_col].astype(str).to_numpy(); n_within = 0; rows = []
    for i in range(len(ok) - 1):
        hit = (hamming_rows(P[i], P[i + 1:]) <= max_bits) & (hamming_rows(D[i], D[i + 1:]) <= max_bits)
        if not hit.any(): continue
        js = np.nonzero(hit)[0] + i + 1
        same = g[js] == g[i]
        n_within += int(same.sum())
        rows += [(g[i], g[j]) for j in js[~same]]
    return n_within, pd.DataFrame(rows, columns=['group_a', 'group_b'])

def link_table(across):
    if across.empty: return pd.DataFrame(columns=['a', 'b', 'pairs'])
    ab = pd.DataFrame({'a': across.min(1), 'b': across.max(1)})
    return ab.groupby(['a', 'b']).size().rename('pairs').reset_index().sort_values('pairs', ascending=False)

class UnionFind:
    def __init__(self, items): self.p = {x: x for x in items}
    def find(self, x):
        while self.p[x] != x: self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

def iou_xywh(a, b):
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0])); ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih; u = a[2] * a[3] + b[2] * b[3] - inter
    return inter / u if u > 0 else 0

LOG = []   # one row per removed / changed item
def log(dataset, item, reason, action, detail=''):
    LOG.append({'dataset': dataset, 'item': str(item), 'reason': reason, 'action': action, 'detail': str(detail)})
SUMMARY = {'seed': SEED, 'created': time.strftime('%Y-%m-%d %H:%M'), 'version': VERSION,
           'thresholds': {'MIN_CROP_SIDE': MIN_CROP_SIDE, 'DUP_BOX_IOU': DUP_BOX_IOU, 'NEAR_DUP_BITS': NEAR_DUP_BITS,
                          'MIN_PAIRS_TO_LINK': MIN_PAIRS_TO_LINK, 'UAVDT_FRAME_STEP': UAVDT_FRAME_STEP,
                          'CROP_MAX_OCCLUSION': CROP_MAX_OCCLUSION, 'CROP_MAX_OUT_OF_VIEW': CROP_MAX_OUT_OF_VIEW}}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 3.4 MB/s eta 0:00:00


## 4. SeaDronesSee: load train + val annotations as one pool

The official train/val split shares 34 of 39 videos (Step 1), so it is not used; Step 3 re-splits by video. The official test set has no labels and is not used for classification.

In [ ]:
META_KEYS = {'altitude': ['altitude', 'height_above_takeoff(meter)', 'height_above_takeoff', 'height'],
             'gimbal_pitch': ['gimbal_pitch', 'gimbal_pitch(degrees)'],
             'date_time': ['date_time', 'datetime', 'time']}
def meta_get(meta, kind):
    for k in META_KEYS[kind]:
        v = (meta or {}).get(k)
        if v not in (None, '', 'nan'):
            try: return float(v) if kind != 'date_time' else str(v)
            except (TypeError, ValueError): return None
    return None

files_on_disk = {p.name: p for p in Path(SDS_LOCAL).rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}}
jsons = {s: next(Path(SDS_LOCAL).rglob(f'instances_{s}.json')) for s in ('train', 'val')}
CATS = None; img_rows, ann_rows = [], []
for split, jp in jsons.items():
    J = json.load(open(jp)); CATS = CATS or {c['id']: c['name'] for c in J['categories']}
    for im in J['images']:
        src, meta = im.get('source') or {}, im.get('meta') or {}
        img_rows.append({'image_id': im['id'], 'file_name': im['file_name'], 'json_w': im['width'], 'json_h': im['height'],
                         'official_split': split, 'video': src.get('video') or src.get('folder_name'),
                         'drone': src.get('drone'), 'frame_no': src.get('frame_no'),
                         'altitude_m': meta_get(meta, 'altitude'), 'gimbal_pitch_deg': meta_get(meta, 'gimbal_pitch'),
                         'date_time': meta_get(meta, 'date_time'), 'meta': meta, 'source': src})
    for a in J['annotations']:
        ann_rows.append({'ann_id': a['id'], 'image_id': a['image_id'], 'category_id': a['category_id'],
                         'x': a['bbox'][0], 'y': a['bbox'][1], 'w': a['bbox'][2], 'h': a['bbox'][3]})
sds_img = pd.DataFrame(img_rows); sds_ann = pd.DataFrame(ann_rows)
assert sds_img.image_id.is_unique, 'image ids repeat across train/val json'
sds_img['path'] = sds_img.file_name.map(lambda f: str(files_on_disk.get(Path(f).name, '')))
print(f'pool: {len(sds_img)} images, {len(sds_ann)} boxes, categories {CATS}')
print('images whose file is missing:', int((sds_img.path == '').sum()))
SUMMARY['sds_before'] = {'images': len(sds_img), 'boxes': len(sds_ann),
                         'boxes_per_class': sds_ann.category_id.map(CATS).value_counts().to_dict()}

pool: 10477 images, 67390 boxes, categories {0: 'ignored', 1: 'swimmer', 2: 'boat', 3: 'jetski', 4: 'life_saving_appliances', 5: 'buoy'}
images whose file is missing: 0


## 5. SeaDronesSee: scan every image (decode + md5 + hashes)

In [ ]:
sds_scan = scan(sds_img.loc[sds_img.path != '', 'path'], label='SDS')
sds_img = sds_img.merge(sds_scan, on='path', how='left')
sds_img['readable'] = sds_img.readable.fillna(False).astype(bool)
sds_img.drop(columns=['meta', 'source']).to_csv(f'{PROC_DIR}/SeaDronesSee/{out_name("SDSImageScan", "csv")}', index=False)

  SDS 2000/10477  61s
  SDS 4000/10477  121s
  SDS 6000/10477  168s
  SDS 8000/10477  228s
  SDS 10000/10477  303s
  SDS 10477/10477  321s


## 6. SeaDronesSee: corrupt and missing files, exact duplicates

In [ ]:
bad = sds_img[~sds_img.readable]
for r in bad.itertuples(): log('SeaDronesSee', r.file_name, 'unreadable or missing file', 'image removed', r.error)
print('unreadable/missing images:', len(bad))
sds_img = sds_img[sds_img.readable].copy()

# exact duplicates: keep the lowest image_id of each md5
sds_img = sds_img.sort_values('image_id')
dup = sds_img[sds_img.duplicated('md5', keep='first')]
keep_of = sds_img.drop_duplicates('md5').set_index('md5').image_id
uf_sds = UnionFind(sorted(sds_img.video.dropna().unique()))
for r in dup.itertuples():
    k = keep_of[r.md5]; kv = sds_img.loc[sds_img.image_id == k, 'video'].iloc[0]
    log('SeaDronesSee', r.file_name, 'exact duplicate file (md5)', 'image removed', f'same bytes as image_id {k} ({kv})')
    if r.video and kv and r.video != kv: uf_sds.union(r.video, kv)
print('exact duplicate images:', len(dup), '| of which across two videos:',
      int(sum(1 for r in dup.itertuples() if r.video != sds_img.loc[sds_img.image_id == keep_of[r.md5], 'video'].iloc[0])))
sds_img = sds_img.drop(dup.index)
sds_ann = sds_ann[sds_ann.image_id.isin(sds_img.image_id)].copy()

unreadable/missing images: 0
exact duplicate images: 0 | of which across two videos: 0


## 7. SeaDronesSee: near duplicates across videos

Open-water frames can look alike even when they come from different flights, so a video pair is linked only when **both** hashes match on at least `MIN_PAIRS_TO_LINK` frame pairs. Check the table: if two linked videos are clearly different flights, raise `MIN_PAIRS_TO_LINK` or lower `NEAR_DUP_BITS` and re-run.

In [ ]:
n_within, across = near_dup_pairs(sds_img, 'video')
print(f'near-identical frame pairs: {n_within} inside one video (expected), {len(across)} across videos')
link = link_table(across)
link.to_csv(f'{PROC_DIR}/SeaDronesSee/{out_name("SDSNearDupVideoPairs", "csv")}', index=False)
display(link.head(30))
for r in link[link.pairs >= MIN_PAIRS_TO_LINK].itertuples():
    uf_sds.union(r.a, r.b); log('SeaDronesSee', f'{r.a} + {r.b}', 'near-identical frames in two videos', 'videos linked into one split group', f'{r.pairs} pairs')
sds_img['split_group'] = sds_img.video.map(lambda v: uf_sds.find(v) if v in uf_sds.p else f'novideo_{v}')
print('videos:', sds_img.video.nunique(), '-> split groups:', sds_img.split_group.nunique())

near-identical frame pairs: 8740 inside one video (expected), 0 across videos


,a,b,pairs


videos: 39 -> split groups: 39


## 8. SeaDronesSee: boxes and the `ignored` category

- `ignored` (id 0): kept as `iscrowd=1` regions for the detector to ignore; never a classification crop.
- If an image file is smaller or larger than the size in the annotation (compressed release), boxes are rescaled to the file.

In [ ]:
IGNORED_ID = next((k for k, v in CATS.items() if v == 'ignored'), 0)
sz = sds_img.set_index('image_id')[['json_w', 'json_h', 'file_w', 'file_h']]
a = sds_ann.join(sz, on='image_id')
a['sx'], a['sy'] = a.file_w / a.json_w, a.file_h / a.json_h
resized = sz[(sz.json_w != sz.file_w) | (sz.json_h != sz.file_h)]
print('images whose file size differs from the annotation:', len(resized))
if len(resized): display(resized.assign(ratio_w=resized.file_w / resized.json_w).ratio_w.round(3).value_counts().head())
for c, s in (('x', 'sx'), ('w', 'sx'), ('y', 'sy'), ('h', 'sy')): a[c] = a[c] * a[s]
for r in resized.itertuples(): log('SeaDronesSee', r.Index, 'file size differs from annotation', 'boxes rescaled', f'{r.json_w}x{r.json_h} -> {r.file_w}x{r.file_h}')

n0 = len(a)
degenerate = a[(a.w <= 0) | (a.h <= 0)]
for r in degenerate.itertuples(): log('SeaDronesSee', f'ann {r.ann_id}', 'zero-size box', 'box removed', f'w={r.w} h={r.h}')
a = a.drop(degenerate.index)

# clip to the image
x2, y2 = (a.x + a.w).clip(upper=a.file_w), (a.y + a.h).clip(upper=a.file_h)
x1, y1 = a.x.clip(lower=0), a.y.clip(lower=0)
clipped = (x1 != a.x) | (y1 != a.y) | (x2 != a.x + a.w) | (y2 != a.y + a.h)
area0 = a.w * a.h
a['x'], a['y'], a['w'], a['h'] = x1, y1, (x2 - x1), (y2 - y1)
gone = a[(a.w <= 0) | (a.h <= 0)]
for r in gone.itertuples(): log('SeaDronesSee', f'ann {r.ann_id}', 'box entirely outside image', 'box removed')
a = a.drop(gone.index); clipped = clipped.loc[a.index]
a['truncated'] = clipped & ((a.w * a.h) < 0.5 * area0.loc[a.index])
for r in a[clipped].itertuples(): log('SeaDronesSee', f'ann {r.ann_id}', 'box crosses image border', 'box clipped')

# duplicate boxes: same image, same class, IoU > DUP_BOX_IOU
drop = []
for (iid, cid), g in a.groupby(['image_id', 'category_id']):
    if len(g) < 2: continue
    b = g[['x', 'y', 'w', 'h']].to_numpy(); ids = g.index.to_numpy()
    for i in range(len(b)):
        if ids[i] in drop: continue
        for j in range(i + 1, len(b)):
            if ids[j] not in drop and iou_xywh(b[i], b[j]) > DUP_BOX_IOU: drop.append(ids[j])
for k in drop: log('SeaDronesSee', f'ann {a.loc[k, "ann_id"]}', 'duplicate box', 'box removed')
a = a.drop(drop)

a['category'] = a.category_id.map(CATS)
a['ignored'] = a.category_id == IGNORED_ID
a['crop_ok'] = (~a.ignored) & (a[['w', 'h']].min(1) >= MIN_CROP_SIDE) & (~a.truncated)
sds_ann = a.drop(columns=['json_w', 'json_h', 'file_w', 'file_h', 'sx', 'sy'])
print(f'boxes: {n0} -> {len(sds_ann)} | ignored regions: {int(sds_ann.ignored.sum())} | '
      f'too small for a crop (<{MIN_CROP_SIDE}px): {int(((sds_ann[["w","h"]].min(1) < MIN_CROP_SIDE) & ~sds_ann.ignored).sum())}')
display(sds_ann.groupby('category').agg(boxes=('ann_id', 'size'), crop_ok=('crop_ok', 'sum')))

images whose file size differs from the annotation: 0
boxes: 67390 -> 67367 | ignored regions: 0 | too small for a crop (<8px): 1


,boxes,crop_ok
category,,
boat,15232,15232
buoy,4945,4945
jetski,2650,2650
life_saving_appliances,1253,1253
swimmer,43287,43286


## 9. SeaDronesSee: altitude (fixed key) and the table per split group

In [ ]:
print(f"altitude filled: {sds_img.altitude_m.notna().mean():.1%} of images "
      f"({sds_img.altitude_m.notna().sum()} / {len(sds_img)})")
alt_by_video = sds_img.groupby('video').altitude_m.agg(['count', 'median', 'min', 'max']).round(1)
display(alt_by_video.head(40))

crops = sds_ann[sds_ann.crop_ok].merge(sds_img[['image_id', 'split_group', 'video']], on='image_id')
sds_groups = (crops.pivot_table(index='split_group', columns='category', values='ann_id', aggfunc='size', fill_value=0)
                   .join(sds_img.groupby('split_group').agg(images=('image_id', 'size'),
                                                            videos=('video', lambda s: ','.join(sorted(set(s)))),
                                                            altitude_median=('altitude_m', 'median'))))
sds_groups.to_csv(f'{PROC_DIR}/{out_name("SDSGroups", "csv")}')
print('groups that contain each class (classification crops):')
display((sds_groups[[c for c in CATS.values() if c in sds_groups and c != 'ignored']] > 0).sum())

altitude filled: 69.7% of images (7305 / 10477)


,count,median,min,max
video,,,,
DJI_0001.mov,15,30.1,29.9,30.2
DJI_0001_d3.mov,94,50.0,49.8,50.2
DJI_0003.mov,60,30.1,29.7,30.4
DJI_0003_d3.mov,1,40.1,40.1,40.1
DJI_0006_d3.mov,94,56.0,55.7,61.0
DJI_0010_d3.mov,113,30.0,29.8,30.2
DJI_0011_d3.mov,394,9.3,4.9,17.4
DJI_0032.MP4,136,9.9,9.6,10.1
DJI_0038.MP4,40,40.1,39.9,40.2


groups that contain each class (classification crops):


,0
swimmer,35
boat,37
jetski,17
life_saving_appliances,9
buoy,18


## 10. SeaDronesSee: write the cleaned COCO file

In [ ]:
def coco_image(r):
    return {'id': int(r.image_id), 'file_name': r.file_name, 'width': int(r.file_w), 'height': int(r.file_h),
            'video': r.video, 'split_group': r.split_group, 'official_split': r.official_split,
            'frame_no': None if pd.isna(r.frame_no) else int(r.frame_no),
            'altitude_m': None if pd.isna(r.altitude_m) else float(r.altitude_m),
            'gimbal_pitch_deg': None if pd.isna(r.gimbal_pitch_deg) else float(r.gimbal_pitch_deg),
            'date_time': r.date_time, 'md5': r.md5}
def coco_ann(r):
    return {'id': int(r.ann_id), 'image_id': int(r.image_id), 'category_id': int(r.category_id),
            'bbox': [round(float(v), 2) for v in (r.x, r.y, r.w, r.h)], 'area': round(float(r.w * r.h), 2),
            'iscrowd': int(r.ignored), 'crop_ok': bool(r.crop_ok), 'truncated': bool(r.truncated)}
clean = {'info': {'description': 'SeaDronesSee ODv2 (compressed), train+val pooled, cleaned by Track 1 Step 2',
                  'created': SUMMARY['created'], 'version': VERSION, 'thresholds': SUMMARY['thresholds']},
         'categories': [{'id': k, 'name': v} for k, v in CATS.items()],
         'images': [coco_image(r) for r in sds_img.itertuples()],
         'annotations': [coco_ann(r) for r in sds_ann.itertuples()]}
sds_out = f'{PROC_DIR}/SeaDronesSee/{out_name("SDSClean", "json")}'
json.dump(clean, open(sds_out, 'w'))
print('saved', sds_out, f"({len(clean['images'])} images, {len(clean['annotations'])} boxes)")
SUMMARY['sds_after'] = {'images': len(sds_img), 'boxes': len(sds_ann), 'ignored_regions': int(sds_ann.ignored.sum()),
                        'crops_per_class': crops.category.value_counts().to_dict(),
                        'videos': int(sds_img.video.nunique()), 'split_groups': int(sds_img.split_group.nunique()),
                        'altitude_fill': round(float(sds_img.altitude_m.notna().mean()), 3),
                        'images_rescaled': int(len(resized)), 'file': sds_out}

saved /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Processed/SeaDronesSee/2026-10-18_SDSClean_v1.json (10477 images, 67367 boxes)


## 11. UAVDT: load GT, ignore regions and attributes

In [ ]:
UAV_CLASSES = {1: 'car', 2: 'truck', 3: 'bus'}
COLS = ['frame', 'target_id', 'x', 'y', 'w', 'h', 'out_of_view', 'occlusion', 'cat']
seq_dirs = {p.name: p for p in Path(UAVDT_LOCAL).rglob('M[0-9][0-9][0-9][0-9]') if p.is_dir()}
gt = {re.match(r'(M\d{4})', p.name).group(1): p for p in Path(UAVDT_LOCAL).rglob('M*_gt_whole.txt')}
ign = {re.match(r'(M\d{4})', p.name).group(1): p for p in Path(UAVDT_LOCAL).rglob('M*_gt_ignore.txt')}
assert gt, 'No *_gt_whole.txt: Step 1 v2 must have fetched UAV-benchmark-MOTD_v1.0.zip'
SEQS = sorted(set(gt) & set(seq_dirs)); print(len(SEQS), 'sequences with images and GT')

uav_box = pd.concat([pd.read_csv(gt[s], header=None, names=COLS).assign(seq=s) for s in SEQS], ignore_index=True)
uav_ign = pd.concat([pd.read_csv(ign[s], header=None, names=COLS).assign(seq=s) for s in SEQS if s in ign], ignore_index=True) \
          if ign else pd.DataFrame(columns=COLS + ['seq'])
uav_box['category'] = uav_box.cat.map(UAV_CLASSES)
print('occlusion levels:', uav_box.occlusion.value_counts().sort_index().to_dict(),
      '| out-of-view levels:', uav_box.out_of_view.value_counts().sort_index().to_dict())
print('ignore regions:', len(uav_ign))
SUMMARY['uavdt_before'] = {'sequences': len(SEQS), 'boxes': len(uav_box),
                           'boxes_per_class': uav_box.category.value_counts().to_dict()}

# sequence attributes (one 0/1 row per sequence); names from the toolkit README if present
attr_files = {re.match(r'(M\d{4})', p.name).group(1): p for p in Path(UAVDT_LOCAL).rglob('M*_attr.txt')}
uav_attr = pd.DataFrame({s: pd.read_csv(p, header=None).iloc[0].to_list() for s, p in attr_files.items()}).T
uav_attr.columns = [f'attr_{i}' for i in range(uav_attr.shape[1])]
print('attribute files:', len(attr_files), '(column meaning: see the UAVDT README; kept as attr_0..n)')

50 sequences with images and GT
occlusion levels: {1: 715470, 2: 9296, 3: 10025, 4: 64004} | out-of-view levels: {1: 725059, 2: 37089, 3: 36647}
ignore regions: 90387
attribute files: 50 (column meaning: see the UAVDT README; kept as attr_0..n)


## 12. UAVDT: scan every frame (decode + md5); hashes only on the 5 fps frames

In [ ]:
frames = pd.DataFrame([{'seq': s, 'frame': int(p.stem.replace('img', '')), 'path': str(p)}
                       for s in SEQS for p in sorted(seq_dirs[s].glob('img*.jpg'))])
frames['keep_5fps'] = (frames.frame - 1) % UAVDT_FRAME_STEP == 0
scan_all = scan(frames.path[~frames.keep_5fps], want_hash=False, label='UAVDT (other frames)')
scan_5 = scan(frames.path[frames.keep_5fps], want_hash=True, label='UAVDT (5 fps)')
frames = frames.merge(pd.concat([scan_all, scan_5]), on='path', how='left')
print('frames:', len(frames), '| at 5 fps:', int(frames.keep_5fps.sum()))

  UAVDT (other frames) 2000/33928  13s
  UAVDT (other frames) 4000/33928  21s
  UAVDT (other frames) 6000/33928  33s
  UAVDT (other frames) 8000/33928  45s
  UAVDT (other frames) 10000/33928  56s
  UAVDT (other frames) 12000/33928  65s
  UAVDT (other frames) 14000/33928  77s
  UAVDT (other frames) 16000/33928  89s
  UAVDT (other frames) 18000/33928  98s
  UAVDT (other frames) 20000/33928  106s
  UAVDT (other frames) 22000/33928  116s
  UAVDT (other frames) 24000/33928  123s
  UAVDT (other frames) 26000/33928  133s
  UAVDT (other frames) 28000/33928  142s
  UAVDT (other frames) 30000/33928  153s
  UAVDT (other frames) 32000/33928  165s
  UAVDT (other frames) 33928/33928  176s
  UAVDT (5 fps) 2000/6807  27s
  UAVDT (5 fps) 4000/6807  51s
  UAVDT (5 fps) 6000/6807  77s
  UAVDT (5 fps) 6807/6807  88s
frames: 40735 | at 5 fps: 6807


## 13. UAVDT: corrupt frames, exact duplicates, near duplicates across sequences

In [ ]:
bad = frames[~frames.readable.astype(bool)]
for r in bad.itertuples(): log('UAVDT', f'{r.seq}/{Path(r.path).name}', 'unreadable frame', 'frame removed', r.error)
print('unreadable frames:', len(bad)); frames = frames[frames.readable.astype(bool)].copy()

uf_uav = UnionFind(SEQS)
d = frames[frames.duplicated('md5', keep=False)].sort_values(['seq', 'frame'])
first = d.drop_duplicates('md5').set_index('md5')
dups = d[d.duplicated('md5', keep='first')]
for r in dups.itertuples():
    k = first.loc[r.md5]
    log('UAVDT', f'{r.seq}/{Path(r.path).name}', 'exact duplicate frame (md5)', 'frame removed', f'same as {k.seq} frame {k.frame}')
    if k.seq != r.seq: uf_uav.union(k.seq, r.seq)
frames = frames.drop(dups.index)
print('exact duplicate frames removed:', len(dups), '| across sequences:', int((dups.seq != first.loc[dups.md5, 'seq'].values).sum()))

f5 = frames[frames.keep_5fps].reset_index(drop=True)
n_within, across = near_dup_pairs(f5, 'seq')
link = link_table(across)
link.to_csv(f'{PROC_DIR}/UAVDT/{out_name("UAVDTNearDupSeqPairs", "csv")}', index=False)
print(f'near-identical 5 fps frame pairs: {n_within} inside a sequence, {len(across)} across sequences')
display(link.head(30))
for r in link[link.pairs >= MIN_PAIRS_TO_LINK].itertuples():
    uf_uav.union(r.a, r.b); log('UAVDT', f'{r.a} + {r.b}', 'near-identical frames in two sequences', 'sequences linked into one split group', f'{r.pairs} pairs')
group_of = {s: uf_uav.find(s) for s in SEQS}
print('sequences:', len(SEQS), '-> split groups:', len(set(group_of.values())))
# Hypothesis to check: sequences sharing the first two digits (M02xx) may come from one flight
by_prefix = pd.Series(SEQS).groupby(pd.Series(SEQS).str[:3]).apply(list)
print('sequences per prefix (Mxx):', by_prefix.map(len).to_dict())

unreadable frames: 0
exact duplicate frames removed: 2 | across sequences: 0
near-identical 5 fps frame pairs: 20952 inside a sequence, 0 across sequences


,a,b,pairs


sequences: 50 -> split groups: 50
sequences per prefix (Mxx): {'M01': 1, 'M02': 10, 'M03': 1, 'M04': 3, 'M05': 1, 'M06': 6, 'M07': 4, 'M08': 2, 'M09': 2, 'M10': 9, 'M11': 2, 'M12': 2, 'M13': 6, 'M14': 1}


## 14. UAVDT: boxes on the 5 fps frames

In [ ]:
W = {r.seq: r.file_w for r in frames.drop_duplicates('seq').itertuples()}
H = {r.seq: r.file_h for r in frames.drop_duplicates('seq').itertuples()}
kept = frames[frames.keep_5fps][['seq', 'frame']]
b = uav_box.merge(kept, on=['seq', 'frame'])
n0 = len(b)
deg = b[(b.w <= 0) | (b.h <= 0)]
for r in deg.itertuples(): log('UAVDT', f'{r.seq} f{r.frame} id{r.target_id}', 'zero-size box', 'box removed')
b = b.drop(deg.index)
b['img_w'], b['img_h'] = b.seq.map(W), b.seq.map(H)
x1, y1 = b.x.clip(lower=0), b.y.clip(lower=0)
x2, y2 = (b.x + b.w).clip(upper=b.img_w), (b.y + b.h).clip(upper=b.img_h)
clipped = (x1 != b.x) | (y1 != b.y) | (x2 != b.x + b.w) | (y2 != b.y + b.h)
area0 = b.w * b.h
b['x'], b['y'], b['w'], b['h'] = x1, y1, x2 - x1, y2 - y1
gone = b[(b.w <= 0) | (b.h <= 0)]
for r in gone.itertuples(): log('UAVDT', f'{r.seq} f{r.frame} id{r.target_id}', 'box entirely outside image', 'box removed')
b = b.drop(gone.index); clipped = clipped.loc[b.index]
for r in b[clipped].itertuples(): log('UAVDT', f'{r.seq} f{r.frame} id{r.target_id}', 'box crosses image border', 'box clipped')
b['truncated'] = clipped & ((b.w * b.h) < 0.5 * area0.loc[b.index])

# boxes whose centre falls in an ignore region of the same frame (frame -1 or 0 in the file = all frames)
b['in_ignore'] = False
if len(uav_ign):
    for (s, fr), g in uav_ign.groupby(['seq', 'frame']):
        m = (b.seq == s) & ((b.frame == fr) | (fr <= 0))
        cx, cy = b.loc[m, 'x'] + b.loc[m, 'w'] / 2, b.loc[m, 'y'] + b.loc[m, 'h'] / 2
        hit = np.zeros(m.sum(), bool)
        for r in g.itertuples(): hit |= ((cx >= r.x) & (cx <= r.x + r.w) & (cy >= r.y) & (cy <= r.y + r.h)).to_numpy()
        b.loc[m, 'in_ignore'] = hit
b['crop_ok'] = ((b[['w', 'h']].min(1) >= MIN_CROP_SIDE) & (b.occlusion <= CROP_MAX_OCCLUSION) &
                (b.out_of_view <= CROP_MAX_OUT_OF_VIEW) & ~b.truncated & ~b.in_ignore)
b['split_group'] = b.seq.map(group_of)
print(f'boxes on 5 fps frames: {n0} -> {len(b)}')
display(b.assign(track=b.seq + '_' + b.target_id.astype(str)).groupby('category')
         .agg(boxes=('target_id', 'size'), crop_ok=('crop_ok', 'sum'), tracks=('track', 'nunique')))

boxes on 5 fps frames: 133460 -> 133460


,boxes,crop_ok,tracks
category,,,
bus,3012,2359,56
car,126243,110783,2523
truck,4205,3442,96


## 15. UAVDT: table per split group, write outputs

In [ ]:
ok = b[b.crop_ok]
boxes_g = ok.pivot_table(index='split_group', columns='category', values='target_id', aggfunc='size', fill_value=0)
tracks_g = (ok.drop_duplicates(['seq', 'target_id', 'category'])
              .pivot_table(index='split_group', columns='category', values='target_id', aggfunc='size', fill_value=0)
              .add_suffix('_tracks'))
uav_groups = boxes_g.join(tracks_g).join(
    frames[frames.keep_5fps].assign(split_group=lambda d: d.seq.map(group_of))
          .groupby('split_group').agg(frames_5fps=('frame', 'size'), sequences=('seq', lambda s: ','.join(sorted(set(s))))))
uav_groups.to_csv(f'{PROC_DIR}/{out_name("UAVDTGroups", "csv")}')
print('groups that contain each class (crop_ok):'); display((boxes_g > 0).sum())

b.drop(columns=['img_w', 'img_h']).to_csv(f'{PROC_DIR}/UAVDT/{out_name("UAVDTBoxes", "csv.gz")}', index=False)
frames.assign(split_group=frames.seq.map(group_of)).drop(columns=['phash', 'dhash']) \
      .to_csv(f'{PROC_DIR}/UAVDT/{out_name("UAVDTFrames", "csv")}', index=False)
if len(uav_attr): uav_attr.rename_axis('seq').to_csv(f'{PROC_DIR}/UAVDT/{out_name("UAVDTAttributes", "csv")}')
if len(uav_ign): uav_ign.to_csv(f'{PROC_DIR}/UAVDT/{out_name("UAVDTIgnoreRegions", "csv")}', index=False)
SUMMARY['uavdt_after'] = {'frames_total': len(frames), 'frames_5fps': int(frames.keep_5fps.sum()), 'boxes_5fps': len(b),
                          'crops_per_class': ok.category.value_counts().to_dict(),
                          'tracks_per_class': ok.drop_duplicates(['seq', 'target_id']).category.value_counts().to_dict(),
                          'sequences': len(SEQS), 'split_groups': len(set(group_of.values())),
                          'groups_with_class': (boxes_g > 0).sum().to_dict()}

groups that contain each class (crop_ok):


,0
category,
bus,23
car,50
truck,33


## 16. Cleaning log and summary

In [ ]:
log_df = pd.DataFrame(LOG, columns=['dataset', 'item', 'reason', 'action', 'detail'])
log_df.to_csv(f'{PROC_DIR}/{out_name("CleaningLog", "csv")}', index=False)
counts = log_df.groupby(['dataset', 'reason', 'action']).size().rename('count').reset_index()
display(counts)
SUMMARY['cleaning_counts'] = counts.to_dict(orient='records')

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if np.isnan(o) else float(o)
    if isinstance(o, float) and o != o: return None
    return o
out = f'{DATA_DIR}/{out_name("Step2Summary", "json")}'
json.dump(jsonable(SUMMARY), open(out, 'w'), indent=2, ensure_ascii=False)
print('saved', out); print(json.dumps(jsonable(SUMMARY), indent=1)[:5000])

,dataset,reason,action,count
0,SeaDronesSee,duplicate box,box removed,23
1,UAVDT,exact duplicate frame (md5),frame removed,2


saved /content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/2026-10-18_Step2Summary_v1.json
{
 "seed": 42,
 "created": "2026-10-18 15:39",
 "version": 1,
 "thresholds": {
  "MIN_CROP_SIDE": 8,
  "DUP_BOX_IOU": 0.95,
  "NEAR_DUP_BITS": 4,
  "MIN_PAIRS_TO_LINK": 3,
  "UAVDT_FRAME_STEP": 6,
  "CROP_MAX_OCCLUSION": 3,
  "CROP_MAX_OUT_OF_VIEW": 2
 },
 "sds_before": {
  "images": 10477,
  "boxes": 67390,
  "boxes_per_class": {
   "swimmer": 43302,
   "boat": 15236,
   "buoy": 4949,
   "jetski": 2650,
   "life_saving_appliances": 1253
  }
 },
 "sds_after": {
  "images": 10477,
  "boxes": 67367,
  "ignored_regions": 0,
  "crops_per_class": {
   "swimmer": 43286,
   "boat": 15232,
   "buoy": 4945,
   "jetski": 2650,
   "life_saving_appliances": 1253
  },
  "videos": 39,
  "split_groups": 39,
  "altitude_fill": 0.697,
  "images_rescaled": 0,
  "file": "/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial/01_Data/Processed/SeaDronesSee/2026-10-18_SDSClean_v1.json"
 },
 "uavdt_befor